# bao.ipynb — Phân lớp KNN và SVM

**Bài tập lớn Khai thác dữ liệu: Dry Bean Dataset**

- Đây là **khung Notebook**, chưa có dữ liệu hoặc kết quả huấn luyện.
- Mỗi thành viên làm trên nhánh riêng **`bao-brand`** do tự tạo sau này; hiện chỉ có nhánh `main` được cập nhật.
- Lưu minh chứng vào `outputs/`, giải thích mọi biểu đồ và chỉ đưa kết quả chạy thật vào báo cáo.
- Nếu dùng Google Colab, bảo đảm dữ liệu được tải lên đúng đường dẫn hoặc chỉnh `ROOT`.


## 0. Thiết lập thư mục và môi trường

Chạy cell dưới đây để xác định đường dẫn; thư mục dữ liệu nguồn chưa có file trong repository.

In [1]:
# Khoi tao du an: thay doi ROOT neu chay notebook tu vi tri khac
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "outputs" / "figures"
TABLES_DIR = ROOT / "outputs" / "tables"
print("Project root:", ROOT)
# TODO: khi co dataset, bo sung cac imports va doc tep can thiet.


Project root: /content


In [2]:
from google.colab import files
from pathlib import Path

# Tạo thư mục dữ liệu
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Tải file Excel lên
uploaded = files.upload()

# Lưu vào thư mục data/raw
for filename, content in uploaded.items():
    file_path = RAW_DIR / "dry_bean.xlsx"
    file_path.write_bytes(content)

    print("Đã lưu:", file_path)

# Kiểm tra
print("Các file trong thư mục raw:")
print(list(RAW_DIR.iterdir()))

Saving dry_bean.xlsx to dry_bean.xlsx
Đã lưu: /content/data/raw/dry_bean.xlsx
Các file trong thư mục raw:
[PosixPath('/content/data/raw/dry_bean.xlsx')]


## 06. Thiết kế bộ train/test và CV

**Nhiệm vụ:** Theo phương án nhóm trưởng chốt: stratify, 10-fold StratifiedKFold, F1-macro; tránh fit scaler trên toàn bộ X trước CV.

**Bài viết của thành viên:**

- Mục tiêu và kiến thức liên quan: _Bổ sung_.
- Phương pháp, tham số và lý do lựa chọn: _Bổ sung_.
- Kết quả chạy thật (bảng/biểu đồ): _Bổ sung_.
- Nhận xét và hạn chế: _Bổ sung_.


In [3]:

# 1. Đọc dữ liệu Dry Bean
import pandas as pd
import numpy as np

df = pd.read_excel(RAW_DIR / "dry_bean.xlsx")

# 2. Kiểm tra dữ liệu
print("Kích thước dữ liệu:", df.shape)
print("Danh sách cột:", df.columns.tolist())

# 3. Tách đặc trưng X và nhãn y
X = df.drop(columns=["Class"])
y = df["Class"]

print("\nKích thước X:", X.shape)
print("Kích thước y:", y.shape)

print("\nSố lớp phân loại:", y.nunique())
print("\nPhân bố các lớp:")
print(y.value_counts())

Kích thước dữ liệu: (13611, 17)
Danh sách cột: ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']

Kích thước X: (13611, 16)
Kích thước y: (13611,)

Số lớp phân loại: 7

Phân bố các lớp:
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

# Chia dữ liệu thành Train và Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Kiểm tra kích thước
print("Kích thước tập Train:", X_train.shape)
print("Kích thước tập Test:", X_test.shape)

print("\nSố lượng nhãn Train:", y_train.shape)
print("Số lượng nhãn Test:", y_test.shape)

# Kiểm tra phân bố lớp
print("\nTỷ lệ lớp trong tập Train:")
print(y_train.value_counts(normalize=True).sort_index().round(4))

print("\nTỷ lệ lớp trong tập Test:")
print(y_test.value_counts(normalize=True).sort_index().round(4))

Kích thước tập Train: (10888, 16)
Kích thước tập Test: (2723, 16)

Số lượng nhãn Train: (10888,)
Số lượng nhãn Test: (2723,)

Tỷ lệ lớp trong tập Train:
Class
BARBUNYA    0.0971
BOMBAY      0.0384
CALI        0.1198
DERMASON    0.2606
HOROZ       0.1416
SEKER       0.1489
SIRA        0.1937
Name: proportion, dtype: float64

Tỷ lệ lớp trong tập Test:
Class
BARBUNYA    0.0973
BOMBAY      0.0382
CALI        0.1197
DERMASON    0.2604
HOROZ       0.1418
SEKER       0.1491
SIRA        0.1935
Name: proportion, dtype: float64


In [5]:
from sklearn.model_selection import StratifiedKFold

# Thiết lập 10-fold Stratified Cross Validation
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

print("Số lượng Fold:", cv.get_n_splits())
print("Phương pháp: StratifiedKFold")
print("Shuffle: True")
print("Random State: 42")

Số lượng Fold: 10
Phương pháp: StratifiedKFold
Shuffle: True
Random State: 42


In [6]:
# Kiểm tra kích thước từng Fold
for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train), start=1
):
    print(
        f"Fold {fold}: "
        f"Train = {len(train_idx)}, "
        f"Validation = {len(val_idx)}"
    )

Fold 1: Train = 9799, Validation = 1089
Fold 2: Train = 9799, Validation = 1089
Fold 3: Train = 9799, Validation = 1089
Fold 4: Train = 9799, Validation = 1089
Fold 5: Train = 9799, Validation = 1089
Fold 6: Train = 9799, Validation = 1089
Fold 7: Train = 9799, Validation = 1089
Fold 8: Train = 9799, Validation = 1089
Fold 9: Train = 9800, Validation = 1088
Fold 10: Train = 9800, Validation = 1088


In [7]:
# Kiểm tra tỷ lệ 7 lớp trong từng Fold
fold_distribution = []

for fold, (_, val_idx) in enumerate(
    cv.split(X_train, y_train), start=1
):
    y_val = y_train.iloc[val_idx]

    proportions = y_val.value_counts(normalize=True)
    proportions.name = f"Fold {fold}"

    fold_distribution.append(proportions)

# Tạo bảng so sánh
fold_df = pd.DataFrame(fold_distribution)

display(fold_df.round(4))

Class,DERMASON,SIRA,SEKER,HOROZ,CALI,BARBUNYA,BOMBAY
Fold 1,0.2608,0.1938,0.1497,0.1414,0.1194,0.0964,0.0386
Fold 2,0.2608,0.1938,0.1488,0.1423,0.1194,0.0964,0.0386
Fold 3,0.2608,0.1938,0.1488,0.1423,0.1194,0.0964,0.0386
Fold 4,0.2599,0.1938,0.1488,0.1414,0.1203,0.0973,0.0386
Fold 5,0.2599,0.1938,0.1488,0.1414,0.1203,0.0973,0.0386
Fold 6,0.2599,0.1938,0.1488,0.1414,0.1203,0.0973,0.0386
Fold 7,0.2608,0.1928,0.1488,0.1414,0.1203,0.0973,0.0386
Fold 8,0.2608,0.1938,0.1488,0.1414,0.1194,0.0973,0.0386
Fold 9,0.2610,0.1939,0.1489,0.1415,0.1195,0.0974,0.0377
Fold 10,0.2610,0.1939,0.1489,0.1415,0.1195,0.0974,0.0377


## 06. Thiết kế bộ Train/Test và Cross Validation

### 1. Mục tiêu và kiến thức liên quan

Mục tiêu của phần này là xây dựng quy trình chia dữ liệu hợp lý để phục vụ huấn luyện và đánh giá mô hình Machine Learning trên bộ dữ liệu Dry Bean.

Dữ liệu gồm 13.611 mẫu, 16 thuộc tính đầu vào và 7 lớp phân loại.

Các kỹ thuật được sử dụng:
- Train/Test Split.
- Stratified Sampling.
- StratifiedKFold Cross Validation.
- Random State.
- F1-macro.

### 2. Phương pháp và tham số lựa chọn

Nhóm sử dụng train_test_split để chia dữ liệu theo tỷ lệ 80% Train và 20% Test.

Các tham số:
- test_size = 0.2
- random_state = 42
- stratify = y

Tiếp theo, sử dụng StratifiedKFold với 10 Fold, shuffle=True và random_state=42 để đánh giá mô hình ổn định hơn.

Stratify giúp duy trì tỷ lệ các lớp trong từng tập dữ liệu, hạn chế tình trạng mất cân bằng khi chia dữ liệu.

### 3. Kết quả thực nghiệm

Kết quả chia dữ liệu:
- Train: 10.888 mẫu.
- Test: 2.723 mẫu.

Quá trình Cross Validation tạo thành 10 Fold:
- Mỗi Fold có khoảng 9.799–9.800 mẫu huấn luyện.
- Mỗi Fold có khoảng 1.088–1.089 mẫu validation.

Kết quả kiểm tra phân bố cho thấy tỷ lệ 7 lớp giữa các Fold tương đối đồng đều.

### 4. Nhận xét và hạn chế

Việc sử dụng StratifiedKFold giúp các Fold duy trì tỷ lệ phân bố lớp tương đối ổn định, hỗ trợ đánh giá mô hình khách quan hơn.

Random State giúp tái lập kết quả thực nghiệm.

Hạn chế: Quá trình chia dữ liệu và Cross Validation chưa phản ánh trực tiếp hiệu suất mô hình. Cần kết hợp Pipeline, GridSearchCV và F1-macro ở bước huấn luyện KNN và SVM để đánh giá hiệu quả phân loại.

## 07. KNN

**Nhiệm vụ:** Pipeline(StandardScaler, KNeighborsClassifier), GridSearchCV tối ưu k/weights; ghi best_params_ và F1-macro 10-fold.

**Bài viết của thành viên:**

- Mục tiêu và kiến thức liên quan: _Bổ sung_.
- Phương pháp, tham số và lý do lựa chọn: _Bổ sung_.
- Kết quả chạy thật (bảng/biểu đồ): _Bổ sung_.
- Nhận xét và hạn chế: _Bổ sung_.


# 07. K-NEAREST NEIGHBORS (KNN)

## 1. Giới thiệu thuật toán KNN

K-Nearest Neighbors (KNN) là thuật toán học máy có giám sát (Supervised Learning), được sử dụng trong các bài toán phân loại (Classification) và hồi quy (Regression).

KNN dự đoán nhãn của một mẫu dữ liệu mới dựa trên K mẫu dữ liệu gần nhất trong tập huấn luyện.

Trong dự án Dry Bean Dataset, KNN được sử dụng để phân loại 7 loại hạt đậu dựa trên 16 thuộc tính hình dạng và kích thước.

## 2. Nguyên lý hoạt động

1. Chọn số lượng láng giềng K.
2. Tính khoảng cách từ mẫu cần dự đoán đến các mẫu trong tập huấn luyện.
3. Sắp xếp khoảng cách theo thứ tự tăng dần.
4. Chọn K mẫu gần nhất.
5. Xác định nhãn dự đoán dựa trên kết quả bỏ phiếu của các láng giềng.

## 3. Các tham số quan trọng

- n_neighbors (K): Số lượng láng giềng được sử dụng để dự đoán.
- weights: Cách tính trọng số cho các láng giềng.
- metric: Phương pháp tính khoảng cách.
- p: Tham số của khoảng cách Minkowski.

## 4. Chuẩn hóa dữ liệu bằng StandardScaler

KNN là thuật toán dựa trên khoảng cách. Nếu các thuộc tính có thang đo khác nhau, thuộc tính có giá trị lớn có thể chi phối kết quả tính khoảng cách.

StandardScaler đưa dữ liệu về trung bình gần 0 và độ lệch chuẩn gần 1.

Việc chuẩn hóa giúp các thuộc tính đóng góp cân bằng hơn vào khoảng cách giữa các mẫu.

## 5. Khoảng cách Euclidean

Khoảng cách Euclidean giữa hai điểm được tính theo công thức:

d(x,y) = sqrt(sum((x_i - y_i)^2))

Khoảng cách càng nhỏ thì hai mẫu dữ liệu càng gần nhau.

## 6. Ưu điểm và hạn chế

### Ưu điểm
- Dễ hiểu và dễ triển khai.
- Không yêu cầu quá trình huấn luyện phức tạp.
- Có thể giải quyết bài toán phân loại nhiều lớp.

### Hạn chế
- Tốn chi phí tính toán khi dữ liệu lớn.
- Nhạy cảm với thang đo thuộc tính.
- Kết quả phụ thuộc vào lựa chọn K và metric.
- Có thể bị ảnh hưởng bởi nhiễu và thuộc tính không liên quan.
## 7. Giải thích các tham số trong mô hình KNN

### 7.1. n_neighbors (K)

Đây là số lượng láng giềng gần nhất được sử dụng để dự đoán nhãn của một mẫu mới.

- K nhỏ: Mô hình nhạy với nhiễu và dễ Overfitting.
- K lớn: Mô hình ổn định hơn nhưng có thể Underfitting.

Trong dự án, K sẽ được lựa chọn thông qua GridSearchCV thay vì chọn tùy ý.

### 7.2. weights

Tham số weights xác định cách tính trọng số cho các láng giềng:

- uniform: Tất cả láng giềng có trọng số như nhau.
- distance: Láng giềng càng gần thì có trọng số càng lớn.

Hai phương pháp sẽ được thử nghiệm để tìm ra lựa chọn phù hợp.

### 7.3. metric

Là phương pháp đo khoảng cách giữa các mẫu dữ liệu.

Dự án sử dụng khoảng cách Minkowski, cho phép lựa chọn các dạng khoảng cách khác nhau thông qua tham số p.

### 7.4. Lý do sử dụng GridSearchCV

GridSearchCV giúp thử nghiệm nhiều tổ hợp tham số KNN và đánh giá bằng Cross Validation.

Tiêu chí lựa chọn mô hình là F1-macro trung bình trên 10 Fold.

Nhờ đó, có thể tìm được bộ tham số phù hợp với bài toán phân loại 7 loại hạt đậu.

In [ ]:
# TODO (bao, 07. KNN): viet code thuc nghiem tai day.
# Luu y: giu nhan Class ra khoi X khi huan luyen/gom cum neu phu hop.


## 09. SVM

**Nhiệm vụ:** Pipeline(StandardScaler, SVC), GridSearchCV thử kernel/C/gamma phù hợp; ghi kết quả CV và phân tích.

**Bài viết của thành viên:**

- Mục tiêu và kiến thức liên quan: _Bổ sung_.
- Phương pháp, tham số và lý do lựa chọn: _Bổ sung_.
- Kết quả chạy thật (bảng/biểu đồ): _Bổ sung_.
- Nhận xét và hạn chế: _Bổ sung_.


In [ ]:
# TODO (bao, 09. SVM): viet code thuc nghiem tai day.
# Luu y: giu nhan Class ra khoi X khi huan luyen/gom cum neu phu hop.


## 07/09. Bàn giao

**Nhiệm vụ:** Bàn giao bảng tham số tối ưu, F1 trung bình, độ lệch (nếu có), hình và notebook cho thong và len.

**Bài viết của thành viên:**

- Mục tiêu và kiến thức liên quan: _Bổ sung_.
- Phương pháp, tham số và lý do lựa chọn: _Bổ sung_.
- Kết quả chạy thật (bảng/biểu đồ): _Bổ sung_.
- Nhận xét và hạn chế: _Bổ sung_.


In [ ]:
# TODO (bao, 07/09. Bàn giao): viet code thuc nghiem tai day.
# Luu y: giu nhan Class ra khoi X khi huan luyen/gom cum neu phu hop.


## Bàn giao và kiểm tra

- [ ] Có giải thích thuật toán/kiến thức từng phần.
- [ ] Code chạy được từ đầu đến cuối (Restart & Run All).
- [ ] Không có số liệu hoặc điểm đánh giá được ghi cứng.
- [ ] Có bảng/biểu đồ và nhận xét từ kết quả thật.
- [ ] Đã push lên nhánh cá nhân và tạo Pull Request để trưởng nhóm xem xét (khi nhóm bắt đầu làm).
